# 202 — Probability-weighted T uncertainty cost

This notebook treats T as uncertain instead of simply wrong by ±10%. It samples a discrete normal distribution around baseline T, re-evaluates the fixed baseline Pareto designs, and calculates probability-weighted cost metrics.

The main project metric is **probability-weighted streamflow shortfall**, which only counts cases where uncertain T gives lower streamflow than the baseline-T prediction.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import uncertainty_project_helpers as up
up.apply_plot_style()

# -----------------------------
# User-editable settings
# -----------------------------
RUN_NAME = "fish_dollars_baseline_0.0_1.0_0.01"
N_TEST_MEMBERS = None
RERUN_REEVALUATION = False

HISTORIC_STREAMFLOW_CFS = 8.6
DEPLETION_OBS_NAME = "lpr:total_combined:bdpl"
PUMPING_COLUMN_FOR_HYDRO_PLOTS = "effective_total_pumping_cfs"

# Probability model for T uncertainty.
T_SIGMA_FRACTION = 0.10       # 10% standard deviation around baseline T
N_T_VALUES = 11               # discrete T sample count
N_SIGMA_EACH_SIDE = 2.0       # range is +/-2 sigma by default
STREAMFLOW_THRESHOLD_CFS = None  # set to a regulatory/management threshold if you want risk-below-threshold

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = up.find_lpr_pycap_opt_dir(NOTEBOOK_DIR)
dirs = up.make_project_dirs(NOTEBOOK_DIR, "202", "probability_weighted_T_uncertainty_cost")
OUTPUT_DIR = dirs["output_dir"]
CACHE_DIR = dirs["cache_dir"]
CACHE_FILE = CACHE_DIR / "202_T_probability_reevaluation_long.csv"
ERROR_CACHE_FILE = CACHE_DIR / "202_T_probability_reevaluation_errors.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CACHE_FILE:", CACHE_FILE)

In [ ]:
# -----------------------------
# Load source-of-truth model context and design set
# -----------------------------
context = up.load_fish_dollars_context(RUN_NAME, PROJECT_DIR)
pareto_archive = up.read_pareto_archive_summary(RUN_NAME, context.run_dir)
pareto_final = up.select_final_feasible_front1(pareto_archive, n_test_members=N_TEST_MEMBERS)
dv_df = up.load_decision_variable_population(context.run_dir)
q_cols = up.get_q_columns(dv_df)

T_probability_table = up.make_discrete_normal_T_table(
    base_T=context.base_T,
    sigma_fraction=T_SIGMA_FRACTION,
    n_values=N_T_VALUES,
    n_sigma_each_side=N_SIGMA_EACH_SIDE,
)
# Use exact baseline label for the T factor closest to 1.0 so the helper can compute baseline-relative metrics.
closest_idx = (T_probability_table["T_factor"] - 1.0).abs().idxmin()
T_probability_table.loc[closest_idx, "scenario"] = "baseline_T"

T_SCENARIOS = {
    row["scenario"]: {
        "T_factor": row["T_factor"],
        "T_value": row["T_value"],
        "probability_weight": row["probability_weight"],
    }
    for _, row in T_probability_table.iterrows()
}

display(T_probability_table)
print("Final feasible front-1 members:", len(pareto_final))
print("Baseline T:", context.base_T)

In [ ]:
# -----------------------------
# Re-evaluate T probability scenarios, or load cached results
# -----------------------------
if CACHE_FILE.exists() and not RERUN_REEVALUATION:
    print("Loading cached probability-weighted re-evaluation:", CACHE_FILE)
    results_long = pd.read_csv(CACHE_FILE)
    error_df = pd.read_csv(ERROR_CACHE_FILE) if ERROR_CACHE_FILE.exists() else pd.DataFrame()
else:
    results_long, error_df = up.reevaluate_designs(
        pareto_final=pareto_final,
        dv_df=dv_df,
        q_cols=q_cols,
        context=context,
        scenarios=T_SCENARIOS,
        historic_streamflow_cfs=HISTORIC_STREAMFLOW_CFS,
        depletion_obs_name=DEPLETION_OBS_NAME,
    )
    results_long.to_csv(CACHE_FILE, index=False)
    if not error_df.empty:
        error_df.to_csv(ERROR_CACHE_FILE, index=False)
    print("Saved cache:", CACHE_FILE)

# Ensure probability weights are present after loading cache.
if "probability_weight" not in results_long.columns or results_long["probability_weight"].isna().all():
    results_long = results_long.drop(columns=["probability_weight"], errors="ignore").merge(
        T_probability_table[["scenario", "probability_weight"]], on="scenario", how="left"
    )

results_long = up.add_known_T_error_metrics(results_long, baseline_label="baseline_T")
member_summary, scenario_summary, overall_summary = up.summarize_probability_weighted_members(
    results_long, streamflow_threshold_cfs=STREAMFLOW_THRESHOLD_CFS
)

display(overall_summary)

In [ ]:
# -----------------------------
# Plots
# -----------------------------
# T probability weights
fig, ax = plt.subplots()
ax.bar(T_probability_table["T_factor"], T_probability_table["probability_weight"], width=0.025, color="0.45")
ax.set_xlabel("T factor relative to baseline")
ax.set_ylabel("Discrete probability weight")
ax.set_title("Discrete normal distribution used for T uncertainty")
up.save_figure(fig, OUTPUT_DIR / "202_T_probability_weights.png")

plot_df = up.sort_for_plot(member_summary, PUMPING_COLUMN_FOR_HYDRO_PLOTS)

# Expected streamflow and uncertainty band
fig, ax = plt.subplots()
ax.plot(plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS], plot_df["baseline_T_streamflow_cfs"], linestyle="--", label="Baseline-T streamflow", color=up.SCENARIO_COLORS["baseline_T"])
ax.plot(plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS], plot_df["expected_streamflow_cfs"], label="Probability-weighted expected streamflow", color=up.SCENARIO_COLORS["expected"])
ax.fill_between(plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS], plot_df["streamflow_p05_cfs"], plot_df["streamflow_p95_cfs"], alpha=0.22, color=up.SCENARIO_COLORS["uncertainty_band"], label="Weighted 5th–95th percentile range")
ax.set_xlabel("Effective total pumping after fish-dollars cutoff (cfs)")
ax.set_ylabel("Streamflow = 8.6 cfs - depletion (cfs)")
ax.set_title("Unknown T: probability-weighted streamflow uncertainty")
ax.legend()
up.save_figure(fig, OUTPUT_DIR / "202_expected_streamflow_with_uncertainty_band.png")

# Probability-weighted absolute streamflow error
fig, ax = plt.subplots()
ax.plot(plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS], plot_df["probability_weighted_absolute_streamflow_error_cfs"], marker="o", markersize=3, color=up.SCENARIO_COLORS["expected"])
ax.set_xlabel("Effective total pumping after fish-dollars cutoff (cfs)")
ax.set_ylabel("Probability-weighted absolute streamflow error (cfs)")
ax.set_title("Expected magnitude of streamflow error from T uncertainty")
up.save_figure(fig, OUTPUT_DIR / "202_probability_weighted_absolute_streamflow_error.png")

# One-sided streamflow shortfall
fig, ax = plt.subplots()
ax.plot(plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS], plot_df["probability_weighted_streamflow_shortfall_cfs"], marker="o", markersize=3, color=up.SCENARIO_COLORS["T_plus_10pct"])
ax.set_xlabel("Effective total pumping after fish-dollars cutoff (cfs)")
ax.set_ylabel("Probability-weighted streamflow shortfall (cfs)")
ax.set_title("One-sided cost of wrongness from T uncertainty")
up.save_figure(fig, OUTPUT_DIR / "202_probability_weighted_streamflow_shortfall.png")

# Example response curves
quantile_positions = [0.10, 0.50, 0.90]
example_members = []
for q in quantile_positions:
    idx = int(round(q * (len(plot_df) - 1)))
    example_members.append(plot_df.iloc[idx]["member"])
example_members = list(dict.fromkeys(example_members))

fig, ax = plt.subplots()
for member in example_members:
    member_df = results_long.loc[results_long["member"] == member].sort_values("T_factor")
    pumping = member_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS].iloc[0]
    ax.plot(member_df["T_factor"], member_df["streamflow_cfs"], marker="o", label=f"member {member}, pumping={pumping:.1f} cfs")
ax.axvline(1.0, linestyle="--", color="0.35")
ax.set_xlabel("T factor relative to baseline")
ax.set_ylabel("Streamflow = 8.6 cfs - depletion (cfs)")
ax.set_title("Example streamflow response curves across uncertain T values")
ax.legend()
up.save_figure(fig, OUTPUT_DIR / "202_example_streamflow_response_curves.png")

In [ ]:
# -----------------------------
# Save outputs
# -----------------------------
results_long.to_csv(OUTPUT_DIR / "202_T_probability_reevaluation_long.csv", index=False)
member_summary.to_csv(OUTPUT_DIR / "202_probability_weighted_member_summary.csv", index=False)
scenario_summary.to_csv(OUTPUT_DIR / "202_T_probability_scenario_summary.csv", index=False)
T_probability_table.to_csv(OUTPUT_DIR / "202_T_probability_weights.csv", index=False)
overall_summary.to_csv(OUTPUT_DIR / "202_probability_weighted_overall_summary.csv", index=False)
if not error_df.empty:
    error_df.to_csv(OUTPUT_DIR / "202_T_probability_reevaluation_errors.csv", index=False)

print("Saved files:")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(" -", f.name)